In [0]:
# Bronze — Source 04: MSK Kafka Clickstream
import sys
sys.path.append('/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared')
from bronze_utils import get_watermark, update_watermark

RAW_BUCKET = 's3://ecommerce-lakehouse-467091806172-raw-01'
SOURCE = '04_clickstream'
TARGET_TABLE = 'bronze.src_04_clickstream.events'
MERGE_KEY = 'event_id'
PATH = f'{RAW_BUCKET}/source=04_kafka_clickstream/clickstream.events/year=*/month=*/day=*/'


In [0]:
from pyspark.sql.functions import col, lit, max as spark_max, from_json
from pyspark.sql.types import *
import json as json_lib

watermark = get_watermark(spark, SOURCE)
print(f'[{SOURCE}] Watermark: {watermark}')

raw_df = spark.read.text(PATH) \
    .filter(col('_metadata.file_modification_time') > lit(watermark))

row_count_raw = raw_df.count()
if row_count_raw == 0:
    print(f'[{SOURCE}] No new files — skipping')
    dbutils.notebook.exit('No new data')

from pyspark.sql.functions import udf

@udf(returnType=StringType())
def unwrap_json(s):
    if s is None: return None
    try:
        inner = json_lib.loads(s)
        if isinstance(inner, str):
            return inner
        import json as j
        return j.dumps(inner)
    except:
        return s

schema = StructType([
    StructField('event_id', StringType()),
    StructField('event_type', StringType()),
    StructField('event_ts', StringType()),
    StructField('session_id', StringType()),
    StructField('user_id', LongType()),
    StructField('anonymous_id', StringType()),
    StructField('page', StringType()),
    StructField('referrer', StringType()),
    StructField('device', StringType()),
    StructField('browser', StringType()),
    StructField('traffic_source', StringType()),
    StructField('ip', StringType()),
    StructField('country', StringType()),
    StructField('product_sku', StringType()),
    StructField('order_id', LongType()),
])

df = raw_df \
    .withColumn('unwrapped', unwrap_json(col('value'))) \
    .withColumn('parsed', from_json(col('unwrapped'), schema)) \
    .select('parsed.*') \
    .filter(col('event_id').isNotNull())

row_count = df.count()
print(f'[{SOURCE}] {row_count} events parsed')

spark.sql('CREATE SCHEMA IF NOT EXISTS bronze.src_04_clickstream')

if spark.catalog.tableExists(TARGET_TABLE):
    from delta.tables import DeltaTable
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(df.alias('s'), f't.{MERGE_KEY} = s.{MERGE_KEY}') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f'MERGE complete')
else:
    df.write.format('delta').mode('overwrite') \
        .option('mergeSchema', 'true').saveAsTable(TARGET_TABLE)
    print(f'Initial load complete')

latest_ts = raw_df.select(spark_max('_metadata.file_modification_time')).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, row_count)
print(f'Watermark updated to {latest_ts}')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'{TARGET_TABLE}: {count} rows')
spark.sql(f"SELECT * FROM bronze.pipeline.watermarks WHERE source = '{SOURCE}'").show()
